### **Sequential Chats and Customer Onboarding**

In [1]:
llm_config = {'model': 'gpt-3.5-turbo'}

#### **Create the needed agents**

In [6]:
from autogen import ConversableAgent

onboarding_personal_information_agent = ConversableAgent(
    name = 'onboarding_personal_information_agent',
    system_message = '''
    You are a helpful customer onboarding agent, you are here to help new customers get started with our product.
    Your job is to gather customer's name and location. Do not ask for other information. Return 'TERMINATE' 
    when you have gathered all the information.''',
    llm_config = llm_config,
    code_execution_config = False,
    human_input_mode = 'NEVER',
)

onboarding_topic_preference_agent = ConversableAgent(
    name = 'onboarding_topic_preference_agent',
    system_message = '''
    You are a helpful customer onboarding agent, you are here to help new customers get started with our product.
    Your job is to gather customer's preferences on news topics. Do not ask for other information.
    Return 'TERMINATE' when you have gathered all the information.''',
    llm_config = llm_config,
    code_execution_config = False,
    human_input_mode = 'NEVER',
)

customer_engagement_agent = ConversableAgent(
    name = 'customer_engagement_agent',
    system_message = '''
    You are a helpful customer service agent here to provide fun for the customer based on the user's
    personal information and topic preferences. This could include fun facts, jokes, or interesting stories.
    Make sure to make it engaging and fun! Return 'TERMINATE' when you are done.''',
    llm_config = llm_config,
    code_execution_config = False,
    human_input_mode = 'NEVER',
    is_termination_msg = lambda msg: 'terminate' in msg.get('content').lower(),
)

customer_proxy_agent = ConversableAgent(
    name = 'customer_proxy_agent',
    llm_config = False,
    code_execution_config = False,
    human_input_mode = 'ALWAYS',
    is_termination_msg = lambda msg: 'terminate' in msg.get('content').lower(),
)

#### **Creating tasks**

In [ ]:
chat_messages = [
    {
        'sender': onboarding_personal_information_agent,
        'recipient': customer_proxy_agent,
        'message': (
            "Hello, I'm here to help you get started with our product."
            "Could you tell me your name and location?"
        ),
        'summary_method': 'reflection_with_llm',
        'summary_args': {
            'summary_prompt': "Return the customer information "
                              "into as JSON object only: "
                              "{'name': '', 'location': ''}",
        },
        'max_turns': 2,
        'clear_history': True
    },
    {
        'sender': onboarding_topic_preference_agent,
        'recipient': customer_proxy_agent,
        'message': (
            "Great! Could you tell me what topics you are "
            "interested in reading about?"
        ),
        'summary_method': 'reflection_with_llm',
        'max_turns': 1,
        'clear_history': False
    },
    {
        'sender': customer_proxy_agent,
        'recipient': customer_engagement_agent,
        'message': "Let's find something fun to read.",
        'max_turns': 1,
        'summary_method': 'reflection_with_llm',
    },
]

#### **Start the onboarding process**

In [8]:
from autogen import initiate_chats

chat_results = initiate_chats(chat_messages)


********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_personal_information_agent (to customer_proxy_agent):

Hello, I'm here to help you get started with our product.Could you tell me your name and location?

--------------------------------------------------------------------------------


c:\Users\hoang\AppData\Local\Programs\Python\Python313\Lib\site-packages\autogen\agentchat\chat.py:57: UserWarning: Repetitive recipients detected: The chat history will be cleared by default if a recipient appears more than once. To retain the chat history, please set 'clear_history=False' in the configuration of the repeating agent.
  warnings.warn(


customer_proxy_agent (to onboarding_personal_information_agent):

Khoa. I am current in a No_name building

--------------------------------------------------------------------------------
onboarding_personal_information_agent (to customer_proxy_agent):

Got it! Could you please provide me with the name of the city or town where you are located?

--------------------------------------------------------------------------------
customer_proxy_agent (to onboarding_personal_information_agent):

Danang

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (abc4adda-91b8-4a97-8f30-5b182f5f9dbd): Maximum turns (2) reached

********************************************************************************
Starting a new chat....

********************************************************************************
onboarding_topic_preference_agent (to customer_proxy_agent):

Great! Could you tell me what topics you are interested in reading about?

#### **Print out the summary**